In [107]:
import pandas as pd
import matplotlib.ticker as mticker
import plotly.express as px
import plotly as py
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import streamlit as st
import plotly.graph_objects as go
from plotly.subplots import make_subplots



In [2]:
cluster_df = pd.read_csv('/Users/nestocr/Documents/DataAnalytics_BootCamp/da-capstone-project-dayane-ernesto/Cleaned Sets/clustering_ready_2019.csv')
cluster_df.head()

,location,prevalence,liters/capita,abstention_rate
0,Albania,0.020138,4.216621,33.654735
1,Andorra,0.026643,10.526667,10.239672
2,Austria,0.038762,11.453333,10.144593
3,Belarus,0.037897,10.250000,12.136096
4,Belgium,0.035462,8.930000,12.199580


In [3]:
scaler = StandardScaler()

In [4]:
features = ['prevalence','liters/capita','abstention_rate']
X=cluster_df[features]

In [5]:
X_scaled=scaler.fit_transform(X)

In [6]:
kmeans=KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_df['cluster'] = kmeans.fit_predict(X_scaled)

In [7]:
cluster_df.groupby('cluster')[['liters/capita', 'abstention_rate', 'prevalence']].mean()


,liters/capita,abstention_rate,prevalence
cluster,,,
0,10.830439,11.429011,0.028788
1,6.968429,16.026404,0.028169
2,4.561409,39.907947,0.023017
3,9.706459,11.147244,0.041155


In [8]:
cluster_names = {
    0: 'Mainstream Drinking Cultures',
    1: 'High Consumption Cultures',
    2: 'Abstainer-Dominant Cultures',
    3: 'High-Risk Drinking Cultures'
}



In [9]:
cluster_df['cluster'] = cluster_df['cluster'].map(cluster_names)

In [10]:
cluster_df1 = pd.read_csv('/Users/nestocr/Documents/DataAnalytics_BootCamp/da-capstone-project-dayane-ernesto/Ernestos_finds/clusters_with_name.csv')

In [11]:
fig = px.scatter_3d(cluster_df1, 
                    x='liters/capita', 
                    y='abstention_rate', 
                    z='prevalence',
                    color='cluster',
                    hover_name='location',
                    title='The 4 Drinking Cultures of Europe (2019)',
                    template='plotly_dark',
                    width=1000,
                    height=1000,    
                    labels={'val': 'Prevalence (Disorders)', 
                            'liters': 'Liters per Capita', 
                            'abstention_rate': '% Abstainers'})
fig.update_layout(
    scene=dict(
        yaxis=dict(autorange='reversed')
    )
)

fig.show()

In [22]:
snapshot = pd.read_csv('/Users/nestocr/Documents/DataAnalytics_BootCamp/da-capstone-project-dayane-ernesto/Cleaned Sets/2019_snapshot_prevalence_of_generations.csv')

In [23]:
snapshot.head()

,location,generation,percent_of_prevalence
0,Albania,Gen_X,0.020352
1,Albania,Gen_Z,0.014243
2,Albania,Millennial,0.025832
3,Czechia,Gen_X,0.032554
4,Czechia,Gen_Z,0.015176


In [24]:
fig = px.bar(
    snapshot,
    x = 'location',
    y = 'percent_of_prevalence',
    color='generation',
    barmode = 'group',
    category_orders={'generation': ['Gen_X', 'Millennial', 'Gen_Z']},  # ← add this
    color_discrete_sequence=px.colors.qualitative.Dark24,  # closest to 'Dark2'
    title = 'The Gen Z pivot: Snapshot 2019 - AUD prevalence by Generation',
    labels = {
        'location':'Countries',
        'percent_of_prevalence':'Average Prevalence of AUD (%)',
        'generation':'Generation' 
    }
)
fig.update_layout(
    yaxis_tickformat='.2f',
    yaxis_ticksuffix='%',
    bargap=0.15,
    bargroupgap=0.05,
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='lightgrey'),
    legend=dict(
        orientation='v',
        yanchor='top',
        y=1,
        xanchor='right',
        x=1.15
    ),
    margin=dict(l=60, r=60, t=60, b=60),
    height=500,
    width=900
)

fig.update_traces(
    marker_line_width=0,
    marker_line_color='rgba(0,0,0,0)'
)
fig.for_each_trace(lambda t: t.update(name={
    'Gen_Z': 'Gen Z (1997-2012)',
    'Millennial': 'Millennial (1981-1996)',
    'Gen_X':'Gen X (1965-1980)'
}.get(t.name, t.name)))

fig.show()

In [53]:
slope_chart = pd.read_csv('/Users/nestocr/Documents/DataAnalytics_BootCamp/da-capstone-project-dayane-ernesto/Cleaned Sets/slope_chart_df_clean.csv')

In [59]:
fig = go.Figure()
offsets = {'Albania': -0.002, 'Czechia': 0.002}

colors = {
    'Albania': '#1f77b4',
    'Czechia': '#ff7f0e',
    'Germany': '#2ca02c',
    'Ireland': '#d62728',
    'Italy': '#9467bd'
}

for country in pivot_peer.index:
    val_2010 = pivot_peer.loc[country, 2010]
    val_2019 = pivot_peer.loc[country, 2019]
    color = colors.get(country, 'grey')

    # Draw line
    fig.add_trace(go.Scatter(
        x=[2010, 2019],
        y=[val_2010, val_2019],
        mode='lines+markers',
        name=country,
        line=dict(color=color, width=2),
        marker=dict(size=8),
        hovertemplate=f'<b>{country}</b><br>Year: %{{x}}<br>Prevalence: %{{y:.2%}}<extra></extra>'
    ))

    # Left label
    fig.add_annotation(
        x=2009.8, y=val_2010,
        text=f'{country}: {val_2010*100:.2f}%',
        xanchor='right', showarrow=False,
        font=dict(size=11, color=color),
        yshift=offsets.get(country,0) *2000
    )

    # Right label
    fig.add_annotation(
        x=2019.2, y=val_2019,
        text=f'{country}: {val_2019*100:.2f}%',
        xanchor='left', showarrow=False,
        font=dict(size=11, color=color),
        yshift=offsets.get(country,0) *2000
    )

fig.update_layout(
    title=dict(text="The '20-Year-Old' Risk Profile: 2010 (Millennial) vs. 2019 (Gen Z)", font=dict(size=14)),
    plot_bgcolor='white',
    showlegend=False,
    height=600,
    width=900,
    xaxis=dict(
        tickvals=[2010, 2019],
        ticktext=['Millennial (in 2010)', 'Gen Z (in 2019)'],
        showgrid=False,
        range=[2007, 2022]
    ),
    yaxis=dict(
        tickformat='.2%',
        showgrid=True,
        gridcolor='lightgrey',
        griddash='dash',
        gridwidth=1,
        range=[0, pivot_peer.values.max() * 1.2]  # 20% headroom above max value
    ),
    margin=dict(l=200, r=200, t=60, b=60)
)

fig.show()

In [62]:
beerdf = pd.read_csv('/Users/nestocr/Documents/DataAnalytics_BootCamp/da-capstone-project-dayane-ernesto/Cleaned Sets/prevalence_vs_beer_clean.csv')

In [63]:
beerdf.head(1)

,location,generation,beer_price_usd,percent_of_prevalence
0,Albania,Gen_Z,0.71,0.014243


In [105]:
colors = {
    'Albania': '#1f77b4',
    'Czechia': '#ff7f0e',
    'Germany': '#2ca02c',
    'Ireland': '#d62728',
    'Italy': '#9467bd'
}
fig = px.scatter(
    
    beerdf, x='beer_price_usd', 
    y='percent_of_prevalence',
    color='location',
    text='location',
    color_discrete_map=colors,
    title='Gen Z Alcohol Risk vs. Beer Price (2019)',
    labels={
        'location':'Country',
        'beer_price_usd': 'Price of 500ml Beer in US dollars',
        'percent_of_prevalence':'Gen Z Prevalence of Disorders (%)',
            },
    size_max=160,
    height=600,
    width=900
)
fig.update_traces(marker=dict(size=20), textposition='top center')
fig.update_traces(
    textposition='bottom center',
    selector={'name':'Albania'}
)
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgrey',griddash='solid'),
    yaxis=dict(showgrid=True, gridcolor='lightgrey', griddash='solid',tickformat='.2%',title=dict(standoff=30)),
    showlegend=False,
    margin=dict(l=100, r=60,t=60, b=60)
)
fig.show()

In [94]:
culturedf = pd.read_csv('/Users/nestocr/Documents/DataAnalytics_BootCamp/da-capstone-project-dayane-ernesto/Cleaned Sets/consumption_vs_prevalence.csv')

In [106]:
colors = {
    'Albania': '#1f77b4',
    'Czechia': '#ff7f0e',
    'Germany': '#2ca02c',
    'Ireland': '#d62728',
    'Italy': '#9467bd'
}
fig = px.scatter(
    
    culturedf, x='liters_per_capita', 
    y='percent_of_prevalence',
    color='location',
    color_discrete_map=colors,
    text='location',
    title='Gen Z Population % with AUDs vs National Consumption in Liters of Pure Alcohol (2019)',
    labels={
        'location':'Country',
        'liters_per_capita': 'Liters of Pure Alcohol Consumed Percapita',
        'percent_of_prevalence':'Gen Z Prevalence of Disorders (%)',
            },
    size_max=160,
    height=600,
    width=900
)
fig.update_traces(marker=dict(size=20), textposition='top center')
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgrey',griddash='dash'),
    yaxis=dict(showgrid=True, gridcolor='lightgrey', griddash='dash',tickformat='.2%',title=dict(standoff=30)),
    showlegend=True,
    margin=dict(l=100, r=60,t=60, b=60)
)
fig.show()

In [108]:
prev_avg = pd.read_csv('/Users/nestocr/Documents/DataAnalytics_BootCamp/da-capstone-project-dayane-ernesto/Cleaned Sets/prevalence_histogram.csv')
con_avg = pd.read_csv('/Users/nestocr/Documents/DataAnalytics_BootCamp/da-capstone-project-dayane-ernesto/Cleaned Sets/consumption_histogram.csv')
abs_avg = pd.read_csv('/Users/nestocr/Documents/DataAnalytics_BootCamp/da-capstone-project-dayane-ernesto/Cleaned Sets/abstention_histogram.csv')

In [115]:
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=['Distribution of Percentage of Population with Alcohol Use Disorders',
                                    'Liters/Capita Distribution',
                                    'Abstention Rate Distribution'])
fig.add_trace(go.Histogram(x=prev_avg['precent_of_prevalence'], nbinsx=10,marker_color='#4182D8'),row=1, col=1)
fig.add_trace(go.Histogram(x=con_avg['NumericValue'], nbinsx=10,marker_color='#C85C8E'), row=1, col=2)
fig.add_trace(go.Histogram(x=abs_avg['NumericValue'], nbinsx=10,marker_color='#4E9B30'), row=1, col=3)


fig.update_layout(
    title='Distribution of Key variables - European Drinking Patterns',
    plot_bgcolor='white',
    showlegend=False,
    bargap=0.05,
    height=400,
    width=1200
)
fig.update_xaxes(showgrid=False)
fig.update_xaxes(title_text= '% of population with AUD', row=1, col=1)
fig.update_xaxes(title_text='Liters of Pure Alcohol', row=1, col=2)
fig.update_xaxes(title_text='% of abstainers in Population', row=1, col=3)
fig.update_yaxes(showgrid=True, gridcolor='lightgrey', title_text='Count of Countries')

In [116]:
gender_comparison = pd.read_csv('/Users/nestocr/Documents/DataAnalytics_BootCamp/da-capstone-project-dayane-ernesto/Cleaned Sets/consumption_males_vs_females.csv')

In [117]:
gender_comparison.head()

,location,sex,age,year,percent_of_prevalence,upper_bound,lower_bound,generation,prevalence %,beer_price_usd,tax_revenue_pct,household_exp_pct,liters_per_capita
0,Albania,Male,15-19 years,2010,0.009399,0.015126,0.005307,Millennial,0.94%,0.71,NaN,1.3,4.216621
1,Albania,Female,15-19 years,2010,0.006388,0.010502,0.003393,Millennial,0.64%,0.71,NaN,1.3,4.216621
2,Albania,Male,15-19 years,2011,0.009435,0.014972,0.005258,Millennial,0.94%,0.71,NaN,1.3,4.216621
3,Albania,Female,15-19 years,2011,0.006384,0.010457,0.003392,Millennial,0.64%,0.71,NaN,1.3,4.216621
4,Albania,Male,15-19 years,2012,0.009483,0.014934,0.005099,Gen_Z,0.95%,0.71,NaN,1.3,4.216621


In [137]:
colors = {
    'Albania': '#1f77b4',
    'Czechia': '#ff7f0e',
    'Germany': '#2ca02c',
    'Ireland': '#d62728',
    'Italy': '#9467bd'
}
fig = px.line(
    gender_comparison,
    x='year',
    y='percent_of_prevalence',
    color='location',
    line_dash='sex',
    symbol='sex',
    title='Gen Z Prevalence : Who is leading the Charge',
    labels={
        'year':'Year',
        'percent_of_prevalence':'Prevalence of Alcohol Use Disorders(%)',
        'location':'Country',
        'sex':'Gender'},
    color_discrete_map=colors,
    markers=True,
    category_orders={'year':sorted(gender_comparison['year'].unique())}
)
fig.update_layout(
    plot_bgcolor='white',
    hovermode='closest',
    legend=dict(
        orientation='v',
        yanchor='top',
        y=1,
        xanchor='left',
        x=1.02
    ),
    width=1000,
    height=600
)
fig.update_xaxes(showgrid=True, gridcolor='lightgrey', gridwidth=0.5)
fig.update_yaxes(showgrid=True, gridcolor='lightgrey', gridwidth=0.5)
fig.show()

In [127]:
fig = px.line(
    gender_comparison, 
    x='year', 
    y='percent_of_prevalence', 
    color='location',      
    line_dash='sex',       
    symbol='sex',          
    title='Gen Z Gender Convergence',
    # This forces the bright, standard Plotly colors
    color_discrete_sequence=px.colors.qualitative.Plotly, 
    markers=True
)

fig.update_layout(plot_bgcolor='white')
fig.show()